[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [4]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [5]:
import torch

In [20]:
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):

    # step 1: temperature scaling
    logits = logits / temperature

    # step 2: top-k filtering
    if top_k > 0:
        sorted_logits = sorted(logits.tolist(), reverse=True)
        cutoff = sorted_logits[top_k - 1]            # kth largest value
        mask   = logits >= cutoff                    # True for top k
        logits = logits.masked_fill(~mask, float('-inf'))  # kill rest

    # step 3: softmax AFTER top-k filtering
    probs = torch.softmax(logits, dim=-1)

    # step 4: top-p filtering
    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumsum = 0.0
        for i in range(len(sorted_probs)):
            cumsum += sorted_probs[i].item()         # ✓ add probability not index
            if cumsum > top_p:
                # zero out everything after this point
                probs[sorted_idx[i+1:]] = float('-inf')
                break
        probs = torch.softmax(probs, dim=-1)         # renormalize after top-p

    # step 5: sample one token index
    return torch.multinomial(probs, num_samples=2)



In [41]:
logits = torch.tensor([0.05, 0.02, 2.0, 0.5, 0.7, 0.9, 0.11, 0.15, 0.04, 0.56, 0.98, 0.76])
print('temp=0:', sample_top_k_top_p(logits.clone(), temperature=0.015))

temp=0: tensor([ 2, 10])


In [9]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

RuntimeError: a Tensor with 2 elements cannot be converted to Scalar

In [7]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (8.0ms)
  ✅ [2/4] Low temperature concentrates (6.0ms)
  ✅ [3/4] All tokens reachable (no filtering) (116.7ms)
  ✅ [4/4] Returns valid index (4.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (135.4ms total)
  Progress saved. Run status() to see your dashboard.

